Softmax <br>
Logits = outputs of the last layer (can be anything) <br>
The logits are fed though softmax and outputs are probabilities <br>
Used because of numerical stability 

<img style="max-width: 400px" src="screens/softmax_formula.png">
<img style="max-width: 400px" src="screens/logits.png">

In [15]:
from IPython.display import display, Math, Latex
display(Math(r'\text{Softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}'))

<IPython.core.display.Math object>

In [11]:
import torch 
import torch.nn as nn 
import numpy as np 

In [16]:
def softmax(x): 
    return np.exp(x) / np.sum(np.exp(x), axis=0) 

x = np.array([2.0, 1.0, 0.01])
outputs = softmax(x) 

f'softmax numpy: {outputs}'

'softmax numpy: [0.66463958 0.24450724 0.09085319]'

In [17]:
x = torch.tensor([2.0, 1.0, 0.01])

# computes softmax along first axix
outputs = torch.softmax(x, dim=0)
outputs

tensor([0.6646, 0.2445, 0.0909])

Cross entropy loss <br>
The better the prediction, the lower the loss. Loss increases as predicted probability diverges from the actual label. <br>
one hot encoding : size(vect) = num_classes; vect(i) = 1 if out_class = i

<img style="max-width: 400px" src="screens/cross_entr.png">

In [18]:
def cross_entropy(actual, predicted): 
    # element-wise, 1/N normalization is not needed
    loss = -np.sum(actual * np.log(predicted))
    return loss

# Y has 3 classes, and the first class is the correct class
Y = np.array([1, 0, 0])

# predicted values as probabilities 
# class 0 has high probability
Y_pred_good = np.array([0.7, 0.2, 0.1])

# class 0 has low probability
Y_pred_bad = np.array([0.1, 0.3, 0.6])

l1 = cross_entropy(Y, Y_pred_good)
l2 = cross_entropy(Y, Y_pred_bad)

f'Loss1 numpy: {l1}, Loss2 numpy: {l2}'

'Loss1 numpy: 0.35667494393873245, Loss2 numpy: 2.3025850929940455'

In [20]:
# native pytorch, already has softmax and cross-entropy loss, NO SOFTMAX NEEDED
# Y not one-hot encoded only class index, Y_pred is in logits form
loss = nn.CrossEntropyLoss()

# class 0 is the correct class, not one-hot encoded
Y = torch.tensor([0]) # can be increased (batched)

# prediction, n_samples x n_classes = 1 x 3, no softmax
Y_pred_good = torch.tensor([[2.0, 1.0, 0.1]]) # can be increased (batched)
Y_pred_bad = torch.tensor([[0.5, 2.0, 0.3]])

l1 = loss(Y_pred_good, Y) 
l2 = loss(Y_pred_bad, Y)

f'Loss1 torch: {l1.item()}, Loss2 torch: {l2.item()}'


'Loss1 torch: 0.4170299470424652, Loss2 torch: 1.840616226196289'

In [21]:
_, predictions1 = torch.max(Y_pred_good, 1) # get predictions along the first dimension 
_, predictions2 = torch.max(Y_pred_bad, 1)

f'Predictions1: {predictions1.item()}, Predictions2: {predictions2.item()}'

'Predictions1: 0, Predictions2: 1'

<img style="width: 400px" src="screens/multiclass.png">
<img style="width: 400px" src="screens/binary.png">
<img style="width: 400px" src="screens/sigmoid.png">

In [22]:
# multiclass classification, ideal for cross entropy loss
class NeuralNetMulticlass(nn.Module): 
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNetMulticlass, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(hidden_size, num_classes) 

    def forward(self, x): 
        out = self.l1(x) 
        out = self.relu(out) 
        out = self.l2(out) 

        # no softmax at the end
        return out
    
model = NeuralNetMulticlass(input_size=28*28, hidden_size=5, num_classes=3)
criterion = nn.CrossEntropyLoss() # applies softmax to output logits automatically 

# for training loop here... 

In [ ]:
# binary classification => implements sigmoid to the output layer 
class NeuralNetBinary(nn.Module): 
    def __init__(self, input_size, hidden_size):
        super(NeuralNetBinary, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        # single output
        self.l2 = nn.Linear(hidden_size, 1)

    def forward(self, x): 
        out = self.l1(x) 
        out = self.relu(out) 
        out = self.l2(out) 

        # sigmoid at the end 
        y_pred = torch.sigmoid(out)
        return y_pred
    
model = NeuralNetBinary(input_size=28*28, hidden_size=5, num_classes=3)
criterion = nn.BCELoss() # binary cross entropy loss

# for training loop here... 